# 이중뮤온 질량 스펙트럼: 실제 CMS 데이터로 입자 "발견"하기

**CERN 탐구 교실 · 업그레이드 U1 · 노트북 1**

이 노트북에서는 CMS 검출기가 2011년에 기록한 **실제 충돌 데이터**(뮤온 두 개가 나온 사건)를 Python으로 분석합니다.

| 단계 | 하는 일 |
|---|---|
| 1 | 데이터 불러오기 (사건 100,000개의 불변질량 M) |
| 2 | 질량 히스토그램 그리기 (보통 축 → 로그–로그 축) |
| 3 | 공명 피크(J/ψ, Υ, Z 등)에 이름 붙이기 |
| 4 | 4-벡터 (E, px, py, pz)로 불변질량을 **직접 계산**해 M 열과 비교 |
| 5 | J/ψ와 Z 피크 확대, 가우스 함수 + 직선 배경으로 맞춤(피팅) → 질량 측정 |
| 6 | 연습 문제 (빈칸 채우기) |

**사용법**: 셀을 클릭하고 `Shift + Enter`를 누르면 그 셀이 실행되고 다음 셀로 넘어갑니다. 위에서부터 차례대로 실행하세요. 처음부터 다시 하려면 메뉴의 `런타임 → 모두 실행`을 누릅니다.

**데이터 출처**: CMS Open Data (opendata.cern.ch, CC0 공개 데이터). 2011년 DoubleMu 데이터에서 뽑은 이중뮤온 사건.

> 그래프 안의 글자는 Colab에 한글 글꼴이 없어서 깨지는 문제를 피하려고 영어로 씁니다.

## 1. 도구 불러오기

- `numpy`: 숫자 배열 계산
- `pandas`: 표(데이터프레임) 다루기 — 엑셀 시트와 비슷합니다
- `matplotlib`: 그래프 그리기

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
print("준비 완료")

## 2. 데이터 불러오기

아래 셀은 이 수업 자료 저장소(GitHub)에서 CSV 파일을 바로 읽습니다. 인터넷이 안 되면 내 컴퓨터의 `data` 폴더에서 읽습니다.

In [ ]:
URL = "https://raw.githubusercontent.com/hkroh99-code/nono99/main/data/cms_dimuon_2011_mass.csv"
URL_EVENTS = "https://raw.githubusercontent.com/hkroh99-code/nono99/main/data/cms_dimuon_2011_events.csv"

def load_csv(url, local_names):
    """인터넷(GitHub)에서 먼저 읽고, 실패하면 내 컴퓨터의 data 폴더에서 읽습니다."""
    try:
        df = pd.read_csv(url)
        print("GitHub에서 읽었습니다:", url)
        return df
    except Exception as err:
        print("인터넷에서 읽지 못했습니다(", type(err).__name__, ") → 로컬 파일을 찾습니다.")
    for path in local_names:
        if os.path.exists(path):
            print("로컬 파일에서 읽었습니다:", path)
            return pd.read_csv(path)
    raise FileNotFoundError("데이터를 찾을 수 없습니다. 인터넷 연결 또는 data 폴더 위치를 확인하세요.")

mass = load_csv(URL, ["../data/cms_dimuon_2011_mass.csv", "data/cms_dimuon_2011_mass.csv"])
print("사건 수:", len(mass))
mass.head()

`M`은 두 뮤온의 **불변질량**(단위 GeV/c², 이하 GeV)입니다. 기초 통계를 보고, 가장 작은 값과 큰 값을 확인하세요.

In [ ]:
M = mass["M"].values          # numpy 배열로 꺼내기
mass["M"].describe()

## 3. 질량 히스토그램 (보통 축)

히스토그램은 값을 구간(bin)으로 나누고, 각 구간에 들어간 사건 수를 막대로 그린 그래프입니다.

In [ ]:
plt.hist(M, bins=240, range=(0, 120), histtype="stepfilled", color="#4c72b0")
plt.xlabel("Dimuon invariant mass M [GeV]")
plt.ylabel("Events / 0.5 GeV")
plt.title("CMS 2011 dimuon events (linear axes)")
plt.show()

**생각해 보기**: 보통 축에서는 낮은 질량 쪽에 사건이 몰려 있어서 높은 질량의 Z 피크(약 91 GeV)가 잘 보이지 않습니다. 0.3 GeV부터 300 GeV까지 **세 자릿수(1000배)** 가 넘는 범위를 한 그래프에 담으려면 어떻게 해야 할까요?

→ **로그 축**을 씁니다. 구간 너비도 로그 간격(`np.logspace`)으로 잡아 질량이 클수록 구간이 넓어지게 합니다.

## 4. 로그–로그 히스토그램과 공명 피크

알려진 입자(공명)의 질량을 점선으로 표시합니다. 각 피크는 **짧은 시간 존재했다가 뮤온 쌍으로 붕괴한 입자**의 흔적입니다.

In [ ]:
RESONANCES = {
    "eta": 0.548, "rho/omega": 0.78, "phi": 1.019,
    "J/psi": 3.097, "psi(2S)": 3.686,
    "Y(1S)": 9.460, "Y(2S)": 10.023, "Y(3S)": 10.355,
    "Z": 91.19,
}

bins = np.logspace(np.log10(0.3), np.log10(300), 400)
counts, edges, _ = plt.hist(M, bins=bins, histtype="step", color="black", linewidth=1)
plt.xscale("log")
plt.yscale("log")
for name, m0 in RESONANCES.items():
    plt.axvline(m0, color="tab:red", linestyle=":", linewidth=1)
    plt.text(m0 * 1.03, counts.max() * 0.6, name, rotation=90, fontsize=8, color="tab:red", va="top")
plt.xlabel("Dimuon invariant mass M [GeV]")
plt.ylabel("Events / bin")
plt.title("CMS 2011 dimuon mass spectrum (log-log)")
plt.show()

**관찰 기록**: 피크 몇 개를 찾았나요? 이름이 붙은 점선과 피크 위치가 잘 맞나요? 활동지 U1의 기록표에 적으세요.

## 5. 4-벡터로 불변질량 직접 계산하기

`cms_dimuon_2011_events.csv`에는 사건 20,000개에 대해 두 뮤온의 에너지와 운동량 성분 (E, px, py, pz, 전하 Q)이 들어 있습니다. 자연 단위(c = 1)에서

$$M = \sqrt{(E_1+E_2)^2 - (p_{x1}+p_{x2})^2 - (p_{y1}+p_{y2})^2 - (p_{z1}+p_{z2})^2}$$

입니다. 이 공식을 코드로 옮겨 봅시다. pandas에서는 **열 전체를 한 번에** 계산할 수 있습니다(반복문이 필요 없음).

In [ ]:
ev = load_csv(URL_EVENTS, ["../data/cms_dimuon_2011_events.csv", "data/cms_dimuon_2011_events.csv"])
print("사건 수:", len(ev))
ev.head()

In [ ]:
E  = ev["E1"]  + ev["E2"]
px = ev["px1"] + ev["px2"]
py = ev["py1"] + ev["py2"]
pz = ev["pz1"] + ev["pz2"]

M2 = E**2 - px**2 - py**2 - pz**2            # 불변질량의 제곱
print("M² < 0 인 사건 수(반올림 오차):", (M2 < 0).sum())
ev["M_calc"] = np.sqrt(M2.clip(lower=0))     # 음수는 0으로 바꾼 뒤 제곱근

diff = (ev["M_calc"] - ev["M"]).abs()
print("계산값과 M 열의 차이: 중앙값 = %.2e GeV, 최댓값 = %.3f GeV" % (diff.median(), diff.max()))
ev[["M", "M_calc"]].head()

거의 모든 사건에서 차이가 0.001 GeV보다 훨씬 작습니다. 차이가 큰 몇몇 사건은 **에너지가 매우 큰 뮤온**(수백 GeV 이상)이 들어 있는 경우입니다. 파일에 저장된 숫자는 소수점 아래 몇 자리에서 반올림되어 있어서, 큰 수끼리 빼는 계산(E² − p²)에서 작은 반올림 오차가 커집니다.

**확인**: 뮤온 하나의 질량 $m = \sqrt{E^2 - p^2}$ 를 계산하면 뮤온 질량(0.1057 GeV) 근처가 나와야 합니다.

In [ ]:
m1 = np.sqrt((ev["E1"]**2 - ev["px1"]**2 - ev["py1"]**2 - ev["pz1"]**2).clip(lower=0))
low_E = ev["E1"] < 20        # 에너지가 작은 뮤온만 (반올림 오차가 작음)
print("뮤온 1의 질량 중앙값 (E1 < 20 GeV): %.4f GeV" % m1[low_E].median())

plt.scatter(ev["M"], ev["M_calc"], s=2, alpha=0.3)
plt.xscale("log"); plt.yscale("log")
plt.xlabel("M (given in file) [GeV]")
plt.ylabel("M (calculated from 4-vectors) [GeV]")
plt.title("Check: calculated vs given invariant mass")
plt.show()

점들이 대각선 위에 놓이면 계산이 맞은 것입니다.

## 6. J/ψ 피크 확대와 맞춤(피팅)

피크 모양을 **가우스 함수(신호) + 직선(배경)** 으로 나타내고, 데이터와 가장 잘 맞는 매개변수를 컴퓨터가 찾게 합니다(최소제곱법). `scipy.optimize.curve_fit`을 씁니다.

$$f(x) = A\,\exp\!\left[-\frac{(x-\mu)^2}{2\sigma^2}\right] + a + b\,x$$

- μ: 피크 중심 = **측정한 입자 질량**
- σ: 피크 폭 (주로 검출기 분해능 때문)
- a + bx: 배경

In [ ]:
from scipy.optimize import curve_fit

def gauss_lin(x, A, mu, sigma, a, b):
    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2) + a + b * x

def fit_peak(lo, hi, mu0, sigma0, nbins=60, title=""):
    """lo~hi 범위를 확대해 히스토그램을 그리고 가우스+직선으로 맞춘다."""
    sel = M[(M > lo) & (M < hi)]
    h, e = np.histogram(sel, bins=np.linspace(lo, hi, nbins + 1))
    x = 0.5 * (e[1:] + e[:-1])
    err = np.sqrt(np.maximum(h, 1))                       # 각 구간의 통계 오차 √N
    p0 = [h.max() - h.min(), mu0, sigma0, h.min(), 0.0]   # 처음 추측값
    popt, pcov = curve_fit(gauss_lin, x, h, p0=p0, sigma=err)
    perr = np.sqrt(np.diag(pcov))
    xx = np.linspace(lo, hi, 400)
    plt.errorbar(x, h, yerr=err, fmt="o", ms=3, color="black", label="CMS data")
    plt.plot(xx, gauss_lin(xx, *popt), color="tab:red", label="Gaussian + linear fit")
    plt.plot(xx, popt[3] + popt[4] * xx, color="tab:gray", linestyle="--", label="background")
    plt.xlabel("M [GeV]"); plt.ylabel("Events / %.3f GeV" % (e[1] - e[0]))
    plt.title(title); plt.legend(); plt.show()
    print("mu    = %.4f ± %.4f GeV" % (popt[1], perr[1]))
    print("sigma = %.4f ± %.4f GeV" % (abs(popt[2]), perr[2]))
    print("범위 안 사건 수 =", len(sel))
    return popt, perr

popt_jpsi, perr_jpsi = fit_peak(2.8, 3.4, 3.1, 0.03, title="J/psi peak")
print("기준값(PDG) 3.097 GeV 와의 차이: %.4f GeV" % (popt_jpsi[1] - 3.097))

## 7. Z 보손 피크

Z 보손은 1983년 CERN에서 처음 발견되었습니다. 기준값은 91.19 GeV입니다.

In [ ]:
popt_z, perr_z = fit_peak(70, 110, 91, 3, nbins=40, title="Z boson peak")
print("기준값 91.19 GeV 와의 차이: %.2f GeV" % (popt_z[1] - 91.19))

**논의**: J/ψ 피크의 σ는 약 0.03 GeV인데 Z 피크의 σ는 약 2 GeV입니다. 이유 두 가지를 생각해 보세요.
1. Z 보손은 수명이 매우 짧아 질량 자체에 퍼짐(자연 폭, 약 2.5 GeV)이 있습니다. J/ψ의 자연 폭은 0.0001 GeV 정도로 아주 좁습니다.
2. 운동량이 큰 뮤온일수록 트랙이 덜 휘어 운동량 측정 오차가 커집니다(주제 4: r = p/(qB)).

또, 가우스 함수는 Z 피크의 실제 모양(브라이트–위그너 분포 + 분해능 + 광자 방출로 생긴 왼쪽 꼬리)을 완벽히 나타내지 못합니다. 그래서 맞춤 결과가 기준값과 조금 다를 수 있습니다.

---
## 8. 연습 문제

빈칸(`None`)을 채우고 셀을 실행하세요. 채우지 않으면 안내 문구만 나옵니다.

**연습 1.** ψ(2S) 피크(약 3.69 GeV)를 맞춤으로 측정하세요. 확대 범위 `lo`, `hi`와 처음 추측값 `mu0`, `sigma0`을 채웁니다.

In [ ]:
lo, hi = None, None          # 예: 3.4, 4.0 처럼 피크 양쪽을 충분히 포함
mu0, sigma0 = None, None     # 예: 3.69, 0.04

if None in (lo, hi, mu0, sigma0):
    print("lo, hi, mu0, sigma0 값을 채운 뒤 다시 실행하세요.")
else:
    popt_psi2s, _ = fit_peak(lo, hi, mu0, sigma0, title="psi(2S) peak")

**연습 2.** Υ(1S) 피크(약 9.46 GeV)를 맞춤으로 측정하세요. 이웃에 Υ(2S), Υ(3S)가 있으므로 범위를 너무 넓게 잡으면 맞춤이 어긋납니다. 어떤 범위가 좋을지 시험해 보세요.

In [ ]:
lo, hi = None, None          # 힌트: Y(2S)는 10.02 GeV
mu0, sigma0 = None, None

if None in (lo, hi, mu0, sigma0):
    print("값을 채운 뒤 다시 실행하세요.")
else:
    popt_ups, _ = fit_peak(lo, hi, mu0, sigma0, title="Upsilon(1S) peak")

**연습 3.** 신호 사건 수 추정: 가우스 함수의 넓이 = A·σ·√(2π) 를 구간 너비로 나누면 피크에 든 **신호 사건 수**입니다. J/ψ 맞춤 결과(`popt_jpsi`)로 계산하세요. 범위 안 전체 사건 수와 비교하면 신호의 비율은?

In [ ]:
bin_width = (3.4 - 2.8) / 60
A, mu, sigma = popt_jpsi[0], popt_jpsi[1], abs(popt_jpsi[2])

n_signal = None              # 여기에 식을 쓰세요: A * sigma * np.sqrt(2 * np.pi) / bin_width

if n_signal is None:
    print("n_signal 식을 채운 뒤 다시 실행하세요.")
else:
    n_total = ((M > 2.8) & (M < 3.4)).sum()
    print("신호 사건 수 ≈ %.0f, 전체 %d, 신호 비율 ≈ %.2f" % (n_signal, n_total, n_signal / n_total))

**연습 4.** 두 뮤온 계의 횡운동량 $p_T = \sqrt{p_x^2 + p_y^2}$ (두 뮤온의 운동량 합)을 계산하고, J/ψ 사건(2.9 < M_calc < 3.3)의 pT 분포를 그려 보세요. pT가 큰 사건과 작은 사건에서 J/ψ 질량이 달라질까요? (불변질량은 관성계에 상관없이 같아야 합니다.)

In [ ]:
ev["pt_pair"] = None         # 여기에 식을 쓰세요: np.sqrt(px**2 + py**2)

if ev["pt_pair"].isna().all():
    print("pt_pair 식을 채운 뒤 다시 실행하세요.")
else:
    j = ev[(ev["M_calc"] > 2.9) & (ev["M_calc"] < 3.3)]
    lowpt, highpt = j[j["pt_pair"] < j["pt_pair"].median()], j[j["pt_pair"] >= j["pt_pair"].median()]
    print("pT 작은 쪽 평균 M = %.4f GeV, pT 큰 쪽 평균 M = %.4f GeV" % (lowpt["M_calc"].mean(), highpt["M_calc"].mean()))
    plt.hist(j["pt_pair"], bins=50, range=(0, 50))
    plt.xlabel("pT of the muon pair [GeV]"); plt.ylabel("Events"); plt.show()

## 9. 정리

활동지(U1)에 다음을 기록하세요.
1. 로그–로그 스펙트럼에서 찾은 공명 피크의 이름과 개수
2. J/ψ, Z 맞춤 결과 (μ ± 오차, σ)와 기준값의 차이
3. 4-벡터로 직접 계산한 질량과 파일의 M 값이 일치했는가? 차이가 큰 사건의 공통점은?
4. 이 분석에서 가장 어려웠던 점과 더 알아보고 싶은 점

다음 노트북: `02_w_charge_ratio.ipynb` — W 보손의 전하 비율과 보이지 않는 중성미자 찾기